Hay que tener instalado la versión de Java 17. Para ello, ejecutar en el terminal:
```bash
sudo apt install -y openjdk-17-jre openjdk-17-jdk
```

Cuando se ejecute en local, asegurarse de que usamos el kernel con el entorno virtual y en el terminal ejecutar:

```bash
pip install pyspark==4.0.0 requests pillow ipython
sudo apt install tree
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
```


In [1]:

!export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
!export SPARK_LOCAL_IP="192.168.0.20"

In [2]:
import requests
import zipfile
import os

# Define the URL of the zip file
zip_url = "https://api.data.neurosys.com:4443/agar-public/AGAR_demo.zip"

# Define the local filename to save the zip file
zip_filename = "AGAR_demo.zip"

# Define the directory to extract the contents into
extraction_directory = "AGAR_demo_extracted"

# Download the zip file
print(f"Downloading {zip_url}...")
try:
    response = requests.get(zip_url, stream=True)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

    with open(zip_filename, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Successfully downloaded {zip_filename}")

    # Unzip the file
    print(f"Extracting {zip_filename} to {extraction_directory}...")
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall(extraction_directory)
    print(f"Successfully extracted contents to {extraction_directory}/")

    # Optional: Clean up the downloaded zip file
    # os.remove(zip_filename)
    # print(f"Removed downloaded zip file: {zip_filename}")

except requests.exceptions.RequestException as e:
    print(f"Error downloading the file: {e}")
except zipfile.BadZipFile:
    print(f"Error: The downloaded file '{zip_filename}' is not a valid zip file.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


Successfully downloaded AGAR_demo.zip
Extracting AGAR_demo.zip to AGAR_demo_extracted...
Successfully extracted contents to AGAR_demo_extracted/


In [3]:
!cd AGAR_demo_extracted && tree

.
└── AGAR_representative
    ├── higher-resolution
    │   ├── bright
    │   │   ├── 1399.jpg
    │   │   ├── 1399.json
    │   │   ├── 349.jpg
    │   │   ├── 349.json
    │   │   ├── 356.jpg
    │   │   ├── 356.json
    │   │   ├── 510.jpg
    │   │   ├── 510.json
    │   │   ├── 518.jpg
    │   │   ├── 518.json
    │   │   ├── 525.jpg
    │   │   ├── 525.json
    │   │   ├── 734.jpg
    │   │   ├── 734.json
    │   │   ├── 735.jpg
    │   │   ├── 735.json
    │   │   ├── 736.jpg
    │   │   ├── 736.json
    │   │   ├── 971.jpg
    │   │   └── 971.json
    │   ├── dark
    │   │   ├── 4826.jpg
    │   │   ├── 4826.json
    │   │   ├── 4904.jpg
    │   │   ├── 4904.json
    │   │   ├── 5206.jpg
    │   │   ├── 5206.json
    │   │   ├── 5207.jpg
    │   │   ├── 5207.json
    │   │   ├── 5212.jpg
    │   │   ├── 5212.json
    │   │   ├── 5270.jpg
    │   │   ├── 5270.json
    │   │   ├── 5271.jpg
    │   │   ├── 5271.json
    │   │   ├── 5308.jpg
    │   │   ├── 5308.json
    │   │   

# Paralelización

In [ ]:
import os
import json
import io
from PIL import Image

# Dimensiones objetivo para el dataset de segmentación
TARGET_WIDTH = 1024
TARGET_HEIGHT = 1024
ORIGINAL_WIDTH = 2048  # Son imágenes cuadradas
ORIGINAL_HEIGHT = 2048

SCALE_X = TARGET_WIDTH / ORIGINAL_WIDTH
SCALE_Y = TARGET_HEIGHT / ORIGINAL_HEIGHT

# Ruta de salida
# output_dir = "/mnt/shared_dataset/output"
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

def logueo(mensaje):
    with open(os.path.join(output_dir, "log.txt"), "a") as log_file:
        log_file.write(f"{mensaje}\n")

def process_image_and_json(row):
    # 🌟 Forzamos la creación de la carpeta antes de abrir el archivo
    os.makedirs("output", exist_ok=True)
    """
    Función que se ejecutará en los ordenadores de los alumnos (Workers).
    Recibe la ruta del JSON y busca su imagen emparejada.
    """
    json_path = row['path']
    # logueo(f"Procesando {row['path']}")

    # 🌟 CORRECCIÓN: Eliminar el protocolo 'file:' si Spark lo añade
    if json_path.startswith("file:"):
        json_path = json_path.replace("file:", "")
    # Reemplazar la extensión para obtener la imagen
    img_path = json_path.replace(".json", ".jpg")

    if not os.path.exists(img_path):
        return f"Error: Imagen no encontrada para {json_path}"

    try:
        # 1. Leer el JSON original
        with open(json_path, 'r') as f:
            data = json.load(f)

        # 2. Cargar la imagen original
        with open(img_path, 'rb') as f:
            img_bytes = f.read()
        img = Image.open(io.BytesIO(img_bytes))

        # --- TAREA 2 & 3: REESCALADO DE IMAGEN Y AJUSTE DE JSON ---
        img_resized = img.resize((TARGET_WIDTH, TARGET_HEIGHT))

        # Estructura para el nuevo JSON modificado
        new_labels = []
        for label in data.get("labels", []):
            # Recalcular las cajas de segmentación proporcionalmente
            new_x = int(label["x"] * SCALE_X)
            new_y = int(label["y"] * SCALE_Y)
            new_w = int(label["width"] * SCALE_X)
            new_h = int(label["height"] * SCALE_Y)

            new_labels.append({
                "id": label["id"],
                "class": label["class"],
                "x": new_x,
                "y": new_y,
                "width": new_w,
                "height": new_h
            })

            # --- TAREA 3: RECORTE (CROP) DE LA COLONIA ---
            # Coordenadas originales para el recorte de máxima calidad
            x, y, w, h = label["x"], label["y"], label["width"], label["height"]
            colony_crop = img.crop((x, y, x + w, y + h))

            # Definir ruta de salida del recorte por clase (Ej: /output/crops/B.subtilis/13895_1.jpg)
            crop_dir = f"{output_dir}/crops/{label['class']}"
            os.makedirs(crop_dir, exist_ok=True)
            crop_filename = f"{data['sample_id']}_{label['id']}.jpg"
            colony_crop.save(os.path.join(crop_dir, crop_filename))

        # Guardar imagen reescalada y nuevo JSON
        seg_img_dir = f"{output_dir}/segmentation/images"
        seg_json_dir = f"{output_dir}/segmentation/labels"
        os.makedirs(seg_img_dir, exist_ok=True)
        os.makedirs(seg_json_dir, exist_ok=True)

        filename_base = os.path.basename(json_path).replace(".json", "")
        img_resized.save(os.path.join(seg_img_dir, f"{filename_base}.jpg"))

        data["labels"] = new_labels
        with open(os.path.join(seg_json_dir, f"{filename_base}.json"), 'w') as f:
            json.dump(data, f, indent=4)

        return f"Procesado con éxito: {filename_base}"

    except Exception as e:
        return f"Error procesando {json_path}: {str(e)}"



In [5]:
import os
os.environ["SPARK_LOCAL_IP"] = "192.168.0.20"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("spark://192.168.0.20:7077") \
    .config("spark.driver.host", "192.168.0.20") \
    .appName("BacteriaDatasetProcessing") \
    .getOrCreate()

print("¡Conectado con éxito al Clúster Distributed!")
# Buscamos todos los archivos .json del dataset
# (Asumiendo que el dataset está montado de forma compartida en la misma ruta para todos)
dataset_path = "AGAR_demo_extracted/AGAR_representative/"

# Leemos los paths usando Spark
json_files_df = spark.read.format("binaryFile") \
    .option("pathGlobFilter", "*.json") \
    .option("recursiveFileLookup", "true") \
    .load(dataset_path) \
    .select("path")

# 🌟 DEBÚGUEO: Verifica cuántos archivos ha detectado Spark antes de procesar
print(f"Número de archivos JSON detectados por Spark: {json_files_df.count()}")

# Convertimos a RDD para procesar fila por fila de manera distribuida
results = json_files_df.rdd.map(process_image_and_json).collect()

# Imprimir resumen en el Driver
for res in results: # Muestra los primeros 10 resultados
    print(res)

spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/21 16:46:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


¡Conectado con éxito al Clúster Distributed!


26/06/21 16:46:24 WARN TaskSetManager: Lost task 0.0 in stage 0.0 (TID 0) (192.168.0.21 executor 0): org.apache.spark.SparkException: [FAILED_READ_FILE.FILE_NOT_EXIST] Encountered error while reading file file:///home/laptop/lab_spark/AGAR_demo_extracted/AGAR_representative/higher-resolution/bright/518.json. File does not exist. It is possible the underlying files have been updated.
You can explicitly invalidate the cache in Spark by running 'REFRESH TABLE tableName' command in SQL or by recreating the Dataset/DataFrame involved. SQLSTATE: KD001
	at org.apache.spark.sql.errors.QueryExecutionErrors$.fileNotExistError(QueryExecutionErrors.scala:831)
	at org.apache.spark.sql.execution.datasources.v2.FileDataSourceV2$.attachFilePath(FileDataSourceV2.scala:140)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:142)
	at scala.collection.Iterator$$anon$9.hasNext(Iterator.scala:583)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$Generate

Py4JJavaError: An error occurred while calling o39.count.
: org.apache.spark.SparkException: [FAILED_READ_FILE.FILE_NOT_EXIST] Encountered error while reading file file:///home/laptop/lab_spark/AGAR_demo_extracted/AGAR_representative/lower-resolution/14380.json. File does not exist. It is possible the underlying files have been updated.
You can explicitly invalidate the cache in Spark by running 'REFRESH TABLE tableName' command in SQL or by recreating the Dataset/DataFrame involved. SQLSTATE: KD001
	at org.apache.spark.sql.errors.QueryExecutionErrors$.fileNotExistError(QueryExecutionErrors.scala:831)
	at org.apache.spark.sql.execution.datasources.v2.FileDataSourceV2$.attachFilePath(FileDataSourceV2.scala:140)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:142)
	at scala.collection.Iterator$$anon$9.hasNext(Iterator.scala:583)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.hashAgg_doAggregateWi

Podemos comparar los tamaños de las imágenes

In [ ]:
!ls "AGAR_demo_extracted/AGAR_representative/lower-resolution/" -lisa

total 6896
438696   4 drwxrwxr-x 1 laptop laptop   4096 jun 21 12:47 .
438653   0 drwxrwxr-x 1 laptop laptop      0 jun 21 12:47 ..
438709 652 -rw-rw-r-- 1 laptop laptop 664895 jun 21 12:47 13895.jpg
438715   4 -rw-rw-r-- 1 laptop laptop   3471 jun 21 12:47 13895.json
438711 620 -rw-rw-r-- 1 laptop laptop 634409 jun 21 12:47 13938.jpg
438700   8 -rw-rw-r-- 1 laptop laptop   5685 jun 21 12:47 13938.json
438713 720 -rw-rw-r-- 1 laptop laptop 733641 jun 21 12:47 14130.jpg
438708   4 -rw-rw-r-- 1 laptop laptop   3337 jun 21 12:47 14130.json
438706 648 -rw-rw-r-- 1 laptop laptop 662886 jun 21 12:47 14380.jpg
438714   8 -rw-rw-r-- 1 laptop laptop   6096 jun 21 12:47 14380.json
438697 724 -rw-rw-r-- 1 laptop laptop 737793 jun 21 12:47 14410.jpg
438701   8 -rw-rw-r-- 1 laptop laptop   6485 jun 21 12:47 14410.json
438704 716 -rw-rw-r-- 1 laptop laptop 732806 jun 21 12:47 14512.jpg
438703   4 -rw-rw-r-- 1 laptop laptop   2804 jun 21 12:47 14512.json
438712 736 -rw-rw-r-- 1 laptop laptop 750725 j

In [ ]:
!ls "AGAR_demo_extracted/AGAR_representative/higher-resolution/vague/" -lisa

total 32352
438675    4 drwxrwxr-x 1 laptop laptop    4096 jun 21 12:47 .
438654    0 drwxrwxr-x 1 laptop laptop       0 jun 21 12:47 ..
438682 3328 -rw-rw-r-- 1 laptop laptop 3407386 jun 21 12:47 11761.jpg
438693   12 -rw-rw-r-- 1 laptop laptop    9491 jun 21 12:47 11761.json
438680 3068 -rw-rw-r-- 1 laptop laptop 3141587 jun 21 12:47 11764.jpg
438695    8 -rw-rw-r-- 1 laptop laptop    7303 jun 21 12:47 11764.json
438676 3128 -rw-rw-r-- 1 laptop laptop 3201924 jun 21 12:47 11773.jpg
438681    4 -rw-rw-r-- 1 laptop laptop    1013 jun 21 12:47 11773.json
438694 3336 -rw-rw-r-- 1 laptop laptop 3414123 jun 21 12:47 11884.jpg
438683    8 -rw-rw-r-- 1 laptop laptop    5475 jun 21 12:47 11884.json
438685 3104 -rw-rw-r-- 1 laptop laptop 3177187 jun 21 12:47 11890.jpg
438678    4 -rw-rw-r-- 1 laptop laptop     653 jun 21 12:47 11890.json
438687 3468 -rw-rw-r-- 1 laptop laptop 3551134 jun 21 12:47 11924.jpg
438684   12 -rw-rw-r-- 1 laptop laptop   10108 jun 21 12:47 11924.json
438686 2972 -rw-r

In [ ]:
!ls "output/segmentation/images" -lisa

total 2868
439096   8 drwxrwxr-x 1 laptop laptop  8192 jun 21 13:12 .
439095   0 drwxrwxr-x 1 laptop laptop     0 jun 21 13:12 ..
439950  64 -rw-rw-r-- 1 laptop laptop 64610 jun 21 13:12 11761.jpg
440346  60 -rw-rw-r-- 1 laptop laptop 60591 jun 21 13:12 11764.jpg
440452  64 -rw-rw-r-- 1 laptop laptop 61964 jun 21 13:12 11773.jpg
439868  72 -rw-rw-r-- 1 laptop laptop 69973 jun 21 13:12 11884.jpg
440555  60 -rw-rw-r-- 1 laptop laptop 57530 jun 21 13:12 11890.jpg
440606  68 -rw-rw-r-- 1 laptop laptop 66329 jun 21 13:12 11924.jpg
439929  64 -rw-rw-r-- 1 laptop laptop 63407 jun 21 13:12 11955.jpg
440153  60 -rw-rw-r-- 1 laptop laptop 60760 jun 21 13:12 12028.jpg
440236  60 -rw-rw-r-- 1 laptop laptop 60741 jun 21 13:12 12031.jpg
440737  64 -rw-rw-r-- 1 laptop laptop 62807 jun 21 13:12 12033.jpg
439210  92 -rw-rw-r-- 1 laptop laptop 92096 jun 21 13:12 13895.jpg
439245  88 -rw-rw-r-- 1 laptop laptop 87294 jun 21 13:12 13938.jpg
440889  92 -rw-rw-r-- 1 laptop laptop 92815 jun 21 13:12 1399.jpg


Comprobamos que en el json, el recorte se ha reducido a la mitad.

In [ ]:
!head -16 AGAR_demo_extracted/AGAR_representative/higher-resolution/vague/11761.json

{
    "background": "vague",
    "classes": [
        "P.aeruginosa",
        "S.aureus"
    ],
    "colonies_number": 54,
    "labels": [
        {
            "class": "S.aureus",
            "height": 39,
            "id": 1,
            "width": 39,
            "x": 706,
            "y": 2112
        },


In [ ]:
!head -16 output/segmentation/labels/11761.json

{
    "background": "vague",
    "classes": [
        "P.aeruginosa",
        "S.aureus"
    ],
    "colonies_number": 54,
    "labels": [
        {
            "id": 1,
            "class": "S.aureus",
            "x": 353,
            "y": 1056,
            "width": 19,
            "height": 19
        },


In [ ]:
# !rm -rf output